# Lab 3.9 &mdash; Real Attention from a Real Model

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Run a real model and ask it for its attention weights
- Search its heads for the one that resolves a pronoun ("it" -> "animal")
- Plot that head as a heatmap and point at the cell that carries the link

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask / attention), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; Self-attention (Q/K/V)](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-03-09")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
Attention is **interpretable**, and different heads **specialise**. We load the real
**distilbert-base-uncased**, ask for `output_attentions=True`, and run a coreference sentence &mdash;
*"the animal did not cross the street because **it** was too tired."* Somewhere in its 6 layers &times; 12
heads is a head that has learned that **"it" refers to "animal"**. We **search every head** for the one
where `it` attends most strongly to `animal`, pull out that head's `seq x seq` matrix, and plot it &mdash;
the *actual* attention the model computed, over its *actual* subword tokens.

> We **search** rather than hardcode a head because the interpretable behaviour lives in *specific*
> heads &mdash; head 0 is usually a "null" head that just dumps attention onto `[SEP]`. (This is why the
> earlier `bert-tiny` demo looked like noise: it is too small to have learned this, and head 0 is a
> `[SEP]` head.)
> The heatmap needs `matplotlib` (already in the lab venv).

## Build it
Turn on attention output, read one head's matrix, and search every head for a learned link.

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModel

def load_attn_model():
    name = "distilbert-base-uncased"   # real, well-trained -- and already used elsewhere in Module 3
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name, attn_implementation="eager", output_attentions=___)   # TODO: True
    model.eval()
    return tok, model

def real_attention(sentence, tok, model, layer, head):
    enc = tok(sentence, return_tensors="pt")
    with torch.no_grad(): out = model(**enc)
    tokens = tok.convert_ids_to_tokens(enc['input_ids'][0])
    A = out.attentions[layer][0, head].numpy()   # ONE (layer, head): a seq x seq matrix
    return tokens, A

def find_head(sentence, query_word, target_word, tok, model):
    """Scan every layer & head; return the (layer, head) whose QUERY token attends
    most to the TARGET token -- i.e. the head that has learned that link."""
    enc = tok(sentence, return_tensors="pt")
    with torch.no_grad(): out = model(**enc)
    tokens = tok.convert_ids_to_tokens(enc['input_ids'][0])
    qi, ti = tokens.index(query_word), tokens.index(target_word)
    best_w, best = -1.0, (0, 0)
    for L in range(len(out.attentions)):
        for H in range(out.attentions[L].shape[1]):
            w = ___   # TODO: out.attentions[L][0, H][qi, ti].item()  -- query->target weight
            if w > best_w: best_w, best = w, (L, H)
    return best, best_w

## Run it for real
Search for the head that resolves the pronoun, then plot it.

In [ ]:
try:
    tok, model = load_attn_model()
    sentence = "the animal did not cross the street because it was too tired"

    # Which head has learned that "it" refers to "animal"?
    (layer, head), w = find_head(sentence, "it", "animal", tok, model)
    print(f"strongest 'it'->'animal' head: layer {layer}, head {head}  (weight {w:.2f})")

    tokens, A = real_attention(sentence, tok, model, layer, head)
    print("tokens:", tokens)
    print("row sums:", np.round(A.sum(axis=1), 2))

    # the pointable evidence: where does the token 'it' look?
    i_it = tokens.index("it")
    order = A[i_it].argsort()[::-1][:4]
    print("'it' attends most to:", ", ".join(f"{tokens[j]}={A[i_it][j]:.2f}" for j in order))

    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 5))
    plt.imshow(A, cmap="viridis")
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right"); plt.yticks(range(len(tokens)), tokens)
    plt.title(f"distilbert attention -- layer {layer}, head {head}\n(the 'it' row lights up on 'animal')")
    plt.colorbar(); plt.tight_layout()
    plt.savefig(WORK + "/real_attention.png", dpi=90); plt.show()
    print("saved:", WORK + "/real_attention.png")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- The winning head (printed above) puts most of the **"it" row's** weight on **"animal"** &mdash; the model
  resolved the pronoun. That is one cell of the heatmap you can explain in plain English.
- Each **row still sums to 1** (it is a softmax) &mdash; the same maths you built by hand in Labs 3.3 and 3.8.
- Heads specialise: some track the next/previous token, many dump attention on `[CLS]`/`[SEP]` (a real,
  well-documented "no-op" behaviour), and a few &mdash; like this one &mdash; carry linguistic structure.
  That is exactly why we **searched** instead of trusting head 0.

## Your turn (open task &mdash; no grader)
Swap in your own sentence and your own `(query, target)` pair &mdash; e.g. a noun and the
adjective that describes it, or a verb and its subject &mdash; and re-run `find_head` to locate the head
that carries *that* link. A "good" answer: you can point to the bright cell and say what it means in plain
English (and, by contrast, name a head that is clearly a "junk"/`[SEP]` head).

---
### Key takeaway
Attention maps are how researchers peek inside transformers. You didn't just read one off a real model -- you searched its heads and found the one that resolved a pronoun. The by-hand maths from Labs 3.3/3.8, made concrete and interpretable.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>